In [ ]:

import pandas as pd
import os


# Read the CSV file
data = pd.read_csv('sql_queries.csv')

# Display the data

total_queries = len(data)
print(total_queries)
data.head()


In [ ]:
# Clean queries that have less than 10 characters

clean_data = data[data['SQL Query'].str.len() >= 10]

clean_data_len = len(clean_data)
print(f"Removed {total_queries - clean_data_len}")


In [ ]:
clean_data2 = clean_data[clean_data['SQL Query'] != 'No value found for "SQL Query" key']
clean_data2_len = len(clean_data2)

print(f"Removed {clean_data_len - clean_data2_len}")


In [ ]:
# Calculate the length of the DataFrame before removing duplicates
clean_data2_len = len(clean_data2)

# Remove duplicates without using inplace
rmv_dupli_data = clean_data2.drop_duplicates()

# Calculate the length of the DataFrame after removing duplicates
rmv_dupli_data_len = len(rmv_dupli_data)

# Print the number of duplicates removed
print(f"Removed {clean_data2_len - rmv_dupli_data_len} duplicates")

In [ ]:
len_data = len(rmv_dupli_data)

print(f"Rows: {len_data}")

rmv_dupli_data.to_csv('sql_queries_clean.csv', index=False)


# Running queries in DB

In [ ]:

# Read the CSV file
queries_df = pd.read_csv('sql_queries_clean.csv')

queries_df.head()

In [ ]:
from logger_manager import LoggerManager
from run_query import get_db_data, init_connection_pool,close_connection_pool, get_db_data2
from print_colors import print_colored

def test_query(query, logger):
    data = get_db_data(query,logger)
    
    if data[0] == 'Database error':
        print("Failed query")
        return False, data[2]

    else:
        print("Sucessful query")
        columns , rows = data
        # Calculate the shape
        num_columns = len(columns)
        num_rows = len(rows)
        shape = (num_rows, num_columns)
        print(f"Data Shape: {shape} ")
        return True , shape
    
logger = LoggerManager.get_logger("SQL_Query_test2").get_logger()
logger.info("TEST 33")

query = queries_df.loc[0, 'SQL Query']
print(query)


init_connection_pool()

valid, description = test_query(query, logger)
print(valid, description)
close_connection_pool()



In [ ]:
import time
import pandas as pd
from logger_manager import LoggerManager
from run_query import get_db_data, init_connection_pool, close_connection_pool
from print_colors import print_colored

def test_query(query, logger):
    data = get_db_data(query, logger)
    
    if data[0] == 'Database error':
        print("Failed query")
        return False, data[2]

    else:
        print("Successful query")
        columns, rows = data
        # Calculate the shape
        num_columns = len(columns)
        num_rows = len(rows)
        shape = (num_rows, num_columns)
        print(f"Data Shape: {shape} ")
        return True, shape

def run_queries_in_batches(queries_df, batch_size=20, wait_time=2):
    # Initialize the connection pool
    init_connection_pool()

    # Prepare results list
    results = []

    logger = LoggerManager.get_logger("SQL_Query_test2").get_logger()
    logger.info("Starting batch query test")

    # Process queries in batches
    for i in range(0, len(queries_df), batch_size):
        batch = queries_df.iloc[i:i + batch_size]

        for index, row in batch.iterrows():
            query = row['SQL Query']
            print(f"Running query: {query}")

            valid, description = test_query(query, logger)
            results.append({
                'SQL Query': query,
                'Valid': valid,
                'Description': description
            })
        

        # Wait between batches
        time.sleep(wait_time)

    # Close the connection pool
    close_connection_pool()

    # Create a new DataFrame from the results
    results_df = pd.DataFrame(results)
    return results_df

# Example usage:
# Assuming queries_df is your DataFrame containing SQL queries
# queries_df = pd.DataFrame({'SQL Query': [...your queries...]})

# Run the queries in batches
results_df = run_queries_in_batches(queries_df)

# Print or save the results DataFrame
print(results_df)
# Optionally, save to a CSV file
results_df.to_csv('sql_query_results.csv', index=True)